# DenseNet201 com validação cruzada

Este caderno registra o experimento da DenseNet201 com validação cruzada estratificada em cinco folds. O objetivo é medir a variação do desempenho da arquitetura antes de compará-la com os outros modelos.

A divisão dos dados segue o mesmo protocolo em todas as rodadas:

- os folds são formados apenas com as imagens reais de `train_real_split.csv`;
- `test_real_split.csv` não participa desta etapa e será usado somente na avaliação final;
- o oversampling é feito depois da separação, apenas sobre o treino do fold;
- o subconjunto de validação conserva a distribuição original;
- não são utilizados DeepSMOTE nem pesos de classe neste experimento.

As probabilidades obtidas nas cinco validações são reunidas em um único arquivo de predições out-of-fold (OOF).

## 1. Preparação do ambiente

In [ ]:
import sys
from pathlib import Path

module_dir = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    direct = candidate / "cross_validation_utils.py"
    nested = candidate / "03_transfer_learning" / "cross_validation" / "cross_validation_utils.py"
    if direct.is_file():
        module_dir = direct.parent
        break
    if nested.is_file():
        module_dir = nested.parent
        break

if module_dir is None:
    raise RuntimeError("Não foi possível localizar cross_validation_utils.py")

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from cross_validation_utils import (
    CLASSES,
    CrossValidationConfig,
    balance_training_dataframe,
    build_fold_assignments,
    configure_runtime,
    fold_distribution,
    load_real_training_dataframe,
    run_cross_validation,
)

print("Módulo carregado de:", module_dir)
print("GPUs disponíveis:", configure_runtime(seed=42))


## 2. Parâmetros do experimento

In [ ]:
config = CrossValidationConfig(
    architecture="densenet201",
    image_size=(320, 320),
    batch_size=12,
    epochs=60,
    n_splits=5,
    learning_rate=1e-4,
    early_stopping_patience=8,
    reduce_lr_patience=4,
    seed=42,
    balance_strategy="random_oversampling",
    oversampling_target_count=None,
    backbone_weights="imagenet",
    save_fold_models=False,
)

config


## 3. Verificação dos dados e dos folds

In [ ]:
train_real_df = load_real_training_dataframe()
fold_assignments = build_fold_assignments(train_real_df, config)

print("Total de imagens reais para validação cruzada:", len(train_real_df))
print("Distribuição geral:")
display(train_real_df["class_name"].value_counts().reindex(CLASSES).to_frame("quantidade"))
print("Distribuição por fold:")
display(fold_distribution(fold_assignments))

example_fold = 1
example_validation = fold_assignments.loc[fold_assignments["fold"] == example_fold]
example_train = fold_assignments.loc[fold_assignments["fold"] != example_fold]
example_oversampled = balance_training_dataframe(example_train, config, fold=example_fold)

print(f"Treino do fold {example_fold} antes do oversampling:")
display(example_train["class_name"].value_counts().reindex(CLASSES).to_frame("quantidade"))
print(f"Treino do fold {example_fold} depois do oversampling:")
display(example_oversampled["class_name"].value_counts().reindex(CLASSES).to_frame("quantidade"))

assert fold_assignments["image"].nunique() == len(fold_assignments)
assert fold_assignments["fold"].between(1, config.n_splits).all()
assert set(example_oversampled["image"]).isdisjoint(example_validation["image"])
assert example_oversampled["label"].value_counts().nunique() == 1
print("Folds sem sobreposição e treino balanceado somente após a separação.")


## 4. Treinamento

Nesta etapa são treinadas cinco instâncias da DenseNet201, uma para cada fold de validação. A execução pode levar várias horas; por isso, vale conferir a distribuição das classes antes de iniciá-la.

In [ ]:
results = run_cross_validation(config)


## 5. Resultados do experimento

In [ ]:
display(results["fold_metrics"])
display(results["summary_metrics"])
print("Métricas OOF:", results["oof_metrics"])
print("Arquivos salvos em:", results["output_dir"])
